In [1]:
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2

import os

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.13.0+cu126
Using device: cuda


In [3]:
transform = v2.Compose([
    v2.ToImage(),
    v2.CenterCrop(1050),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

Import data from ../data/raw/aptos2019-blindness-detection/test_images and split into test and train from this folder

In [4]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        image = Image.open(img_path).convert('RGB')
        label = row['diagnosis']
        if self.transform:
            image = self.transform(image)
        return image, label

In [5]:
data = BlindnessDataset(csv_file='..\\data\\raw\\aptos2019-blindness-detection\\train.csv', img_dir='..\\data\\raw\\aptos2019-blindness-detection\\train_images', transform=transform)
train_size = int(0.8 * len(data))
test_size = len(data) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(data, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [6]:
image, label = train_dataset[0]

In [7]:
print(image.size())

torch.Size([3, 1050, 1050])


In [8]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 12, 5) # (12, 1046, 1046)
        self.pool = nn.MaxPool2d(2, 2) # (12, 523, 523)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 519, 519) -> (24, 259, 259)
        self.fc1 = nn.Linear(24 * 259 * 259, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.fc4 = nn.Linear(10, 5)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [9]:
net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
import gc

flag = 0
for epoch in range(30):
    print(f'Epoch {epoch + 1}')
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        try:
            inputs, labels = data
            optimizer.zero_grad(set_to_none=True)
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            if flag == 0:
                print(outputs.device)
                flag = 1
            print(f"Batch {i + 1}: inputs shape: {inputs.shape}, labels shape: {labels.shape}, outputs shape: {outputs.shape}")
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        except torch.cuda.OutOfMemoryError as e:
            print(f"Out of memory error at batch {i + 1} in epoch {epoch + 1}: {e}")
            e.__traceback__ = None          # drop the traceback holding frame locals
            optimizer.zero_grad(set_to_none=True)
            inputs = labels = outputs = loss = None
            gc.collect()                    # collect the now-unreferenced graph
            torch.cuda.empty_cache()
            continue
        finally:
            # runs on success too, so live tensors don't linger between batches
            inputs = labels = outputs = loss = None

Epoch 1
cuda:0
Batch 1: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 2: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 3: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 4: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 5: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 6: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 7: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
Batch 8: inputs shape: torch.Size([32, 3, 1050, 1050]), labels shape: torch.Size([32]), outputs shape: torch.Size([32, 5])
B

In [12]:
torch.save(net.state_dict(), "keshavs_trained_net.pth")

In [13]:
net = NeuralNet()
net.load_state_dict(torch.load("keshavs_trained_net.pth"))

<All keys matched successfully>

In [17]:
correct = 0
total = 0

net.eval()  # Set the model to evaluation mode
net.to(device)  # Move the model to the appropriate device

with torch.no_grad():  
    for data in test_loader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        print(f'Batch accuracy: {(predicted == labels).sum().item() / labels.size(0) * 100:.2f}%')

print(f'Accuracy of the network on the test images: {100 * correct / total} %')

Batch accuracy: 65.62%
Batch accuracy: 71.88%
Batch accuracy: 78.12%
Batch accuracy: 65.62%
Batch accuracy: 75.00%
Batch accuracy: 65.62%
Batch accuracy: 68.75%
Batch accuracy: 75.00%
Batch accuracy: 68.75%
Batch accuracy: 71.88%
Batch accuracy: 62.50%
Batch accuracy: 68.75%
Batch accuracy: 71.88%
Batch accuracy: 65.62%
Batch accuracy: 65.62%
Batch accuracy: 78.12%
Batch accuracy: 68.75%
Batch accuracy: 71.88%
Batch accuracy: 56.25%
Batch accuracy: 71.88%
Batch accuracy: 50.00%
Batch accuracy: 62.50%
Batch accuracy: 65.52%
Accuracy of the network on the test images: 68.07639836289222 %


In [11]:
# import gc, torch

# # move model off GPU and drop it
# try:
#     net.cpu()
#     del net
# except NameError:
#     pass

# # optimizer holds momentum/Adam state tensors on GPU
# try:
#     optimizer.state.clear()
#     del optimizer
# except NameError:
#     pass

# # drop any leftover training tensors
# for name in ['inputs', 'labels', 'outputs', 'loss', 'data']:
#     globals().pop(name, None)

# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

# print(f'Live tensors: {sum(1 for o in gc.get_objects() if torch.is_tensor(o))}')
# print(f'Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
# print(f'Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB')